# OpsPilot — Streamlit Chat UI Launcher

This notebook starts the Streamlit chat interface as a background process and gives you the correct URL to open it inside Vocareum.

**Run cells in order (1 → 4). Cell 5 is optional for an embedded view. Cell 6 stops the server.**

| Cell | What it does |
|------|--------------|
| 1 | Install Streamlit |
| 2 | Find the project root |
| 3 | Start Streamlit in the background |
| 4 | Show clickable access URLs |
| 5 | Embed the UI directly in this notebook (optional) |
| 6 | Stop the Streamlit server |

In [ ]:
# Cell 1 — Install Streamlit (skip if already installed)
!pip install streamlit pysqlite3-binary -q
print('Streamlit ready ✓')

In [ ]:
# Cell 2 — Locate the project root and the app file
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())

# Walk up until we find streamlit_app.py
PROJECT_ROOT = NOTEBOOK_DIR
for candidate in [NOTEBOOK_DIR, NOTEBOOK_DIR.parent, Path('/voc/work')]:
    if (candidate / 'streamlit_app.py').exists():
        PROJECT_ROOT = candidate
        break

APP_PATH = PROJECT_ROOT / 'streamlit_app.py'

print(f'Project root : {PROJECT_ROOT}')
print(f'App path     : {APP_PATH}')
print(f'File exists  : {APP_PATH.exists()}')

if not APP_PATH.exists():
    print('\n❌  streamlit_app.py not found.')
    print('   Make sure you uploaded it to the project root (same folder as the agent/ directory).')
else:
    print('\n✅  Ready to launch.')

In [ ]:
# Cell 3 — Start Streamlit as a background process
import subprocess
import time

PORT = 8501

# Kill any previous instance on this port
os.system(f"fuser -k {PORT}/tcp 2>/dev/null || pkill -f 'streamlit run' 2>/dev/null || true")
time.sleep(1)

# Launch Streamlit with settings tuned for Vocareum / JupyterHub
proc = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run', str(APP_PATH),
        f'--server.port={PORT}',
        '--server.address=0.0.0.0',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
        '--browser.gatherUsageStats=false',
    ],
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(f'Starting OpsPilot on port {PORT}...  (waiting 6 seconds)')
time.sleep(6)

if proc.poll() is None:
    print(f'✅  Streamlit is running!  PID = {proc.pid}')
    print('   → Run Cell 4 to get your access URL.')
else:
    # Read startup output to help diagnose the problem
    out = proc.stdout.read().decode('utf-8', errors='replace')
    print('❌  Streamlit failed to start. Output:')
    print(out[-2000:])

In [ ]:
# Cell 4 — Show the correct URL to open OpsPilot
import os
import socket
from IPython.display import display, HTML

PORT = 8501

# ── Detect JupyterHub proxy prefix (set automatically on Vocareum) ────────────
hub_prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '')

# ── Detect external IP ────────────────────────────────────────────────────────
try:
    ext_ip = socket.gethostbyname(socket.gethostname())
except Exception:
    ext_ip = None

# ── Build the list of URLs to try ─────────────────────────────────────────────
rows = ''

if hub_prefix:
    proxy_url = f'{hub_prefix}proxy/{PORT}/'
    rows += f'''
    <tr style="background:#e8f5e9">
      <td><b>⭐ JupyterHub Proxy</b><br><small>Best option — works inside Vocareum</small></td>
      <td><a href="{proxy_url}" target="_blank" style="font-size:1.1em">{proxy_url}</a></td>
    </tr>'''
else:
    rows += '''
    <tr style="background:#fff9c4">
      <td colspan="2">⚠️  <code>JUPYTERHUB_SERVICE_PREFIX</code> not set — proxy URL unavailable.<br>
      Try the External IP link below or use port-forwarding.</td>
    </tr>'''

if ext_ip and not ext_ip.startswith('127.'):
    ext_url = f'http://{ext_ip}:{PORT}'
    rows += f'''
    <tr>
      <td><b>🌐 External IP</b><br><small>Open this in a new browser tab</small></td>
      <td><a href="{ext_url}" target="_blank">{ext_url}</a></td>
    </tr>'''

rows += f'''
    <tr>
      <td><b>🖥 Localhost</b><br><small>Works if you have SSH port-forwarding active</small></td>
      <td><a href="http://localhost:{PORT}" target="_blank">http://localhost:{PORT}</a></td>
    </tr>'''

html = f'''
<div style="font-family:sans-serif; max-width:750px">
  <h3>🛡️ OpsPilot Chat UI — Access URLs</h3>
  <table border="1" cellpadding="8" cellspacing="0" style="border-collapse:collapse; width:100%">
    <tr style="background:#1565c0; color:white">
      <th>Method</th><th>URL (click to open)</th>
    </tr>
    {rows}
  </table>
  <p style="color:#555; font-size:0.9em">
    Try the <b>⭐ JupyterHub Proxy</b> link first — it opens inside your existing Vocareum session.<br>
    If the page is blank, append a trailing <code>/</code> to the URL and reload.
  </p>
</div>
'''
display(HTML(html))

In [ ]:
# Cell 5 — Embed OpsPilot directly inside this notebook (optional)
# This works when the JupyterHub proxy URL is available (Cell 4 shows ⭐ link).
# If the iframe is blank, use the clickable URL from Cell 4 instead.

from IPython.display import IFrame, display

hub_prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '')

if hub_prefix:
    proxy_url = f'{hub_prefix}proxy/{PORT}/'
    print(f'Embedding: {proxy_url}')
    display(IFrame(src=proxy_url, width='100%', height=750))
else:
    print('JupyterHub proxy prefix not found.')
    print(f'Open this URL manually in a new browser tab:')
    print(f'  http://localhost:{PORT}')

In [ ]:
# Cell 6 — Stop the Streamlit server
# Run this when you are done using the chat UI.

try:
    proc.terminate()
    proc.wait(timeout=5)
    print(f'✅  Streamlit (PID {proc.pid}) stopped.')
except Exception as e:
    print(f'Note: {e}')
    os.system("pkill -f 'streamlit run' 2>/dev/null || true")
    print('Sent kill signal to any streamlit process.')